# When Your Data Structure Is Your Optimization Landscape

**WQS Binary Search on the Knowledge Substrate**

*A proof-of-concept companion to [A Content-Addressed Adaptive Knowledge Substrate for Distributed Epistemic Coordination](joven_knowledge_substrate.md) (Joven, 2026)*

---

The hello_world notebook demonstrated two things: content-addressed change detection and scoped
recomputation. That was the *correctness* story. This notebook is about *efficiency* — specifically,
the claim from the paper (§2.3, §8.2) that the substrate's structure should make calibration tractable.

Here's the observation that got me out of my chair:

> The Merkle DAG is a tree. Trees unlock DP. DP with a budget constraint unlocks WQS binary search.
> Therefore: every tunable in the substrate that controls "how much compute to spend" is a
> Lagrangian penalty parameter you can binary search over.

This is the [aliens trick](https://codeforces.com/blog/entry/68534) from competitive programming (IOI 2016 "Aliens").
The idea: if you have an optimization problem with a cardinality or budget constraint, and the
optimal value is **concave** in that constraint, you can remove the constraint entirely, add a
per-unit penalty λ, and binary search on λ until the unconstrained solution happens to satisfy
the constraint. The unconstrained problem is almost always easier.

For the substrate, diminishing returns on traversal depth means the gain-vs-budget curve IS concave.
So WQS applies. Not just to one tunable — to all of them. And the tree decomposition means
optimizing them simultaneously is O(n · (log V)^k), not O(n · V^k).

If the math actually works, that's the difference between calibration being a research problem
and calibration being a subroutine.

## The Setup

We extend the hello_world primitives with **depth scores** (§2.2) and **tiered costs** (§2.3).
Each knowledge node now carries:

- **Scoring dimensions**: recency, structural centrality, information potential
- **Tier costs**: what it costs to process this node at Tier 0 (skip), 1 (hash check), 2 (local traversal), 3 (LLM call)
- **Tier gains**: how much information each tier reveals, as a function of the scoring dimensions

The tree structure comes from the DAG. In a general Merkle DAG you'd have a forest
of trees rooted at query entry points; here we work with a single rooted tree for clarity.

In [ ]:
import hashlib
import json
import math
import random
import time
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class KNode:
    """Knowledge node with substrate depth scoring and tiered cost model."""
    nid: str
    content: str
    depth: int
    children: list = field(default_factory=list)

    # Scoring dimensions (§2.2)
    recency: float = 0.0
    centrality: float = 0.0
    info_potential: float = 0.0

    # Tier costs: (skip, hash_check, local_traversal, llm_call)
    tier_costs: tuple = (0.0, 1.0, 8.0, 80.0)

    def depth_score(self, weights=(0.4, 0.3, 0.3)):
        """Composite depth score — determines traversal priority."""
        return (weights[0] * self.recency +
                weights[1] * self.centrality +
                weights[2] * self.info_potential)

    def tier_gain(self, tier, weights=(0.4, 0.3, 0.3)):
        """Information gain at a given tier, parameterized by scoring weights."""
        if tier == 0:
            return 0.0
        base = self.depth_score(weights)
        # Tier 1 reveals 30%, Tier 2 reveals 70%, Tier 3 reveals 100%
        tier_fraction = [0.0, 0.3, 0.7, 1.0][tier]
        return base * tier_fraction

    def content_hash(self):
        return hashlib.sha256(self.content.encode()).hexdigest()[:16]


def generate_tree(max_depth=4, branching=3, seed=42):
    """Generate a tree-shaped knowledge graph with realistic tier costs."""
    rng = random.Random(seed)
    counter = [0]

    def build(d):
        nid = f"n{counter[0]}"
        counter[0] += 1

        info = max(0.05, 1.0 - d * 0.15 + rng.gauss(0, 0.08))
        node = KNode(
            nid=nid, content=f"knowledge_{nid}", depth=d,
            recency=max(0, rng.random()),
            centrality=max(0, 1.0 - d * 0.2 + rng.gauss(0, 0.05)),
            info_potential=info,
            tier_costs=(0, 1, 5 + rng.random() * 5, 50 + rng.random() * 50),
        )
        if d < max_depth:
            for _ in range(rng.randint(1, branching)):
                node.children.append(build(d + 1))
        return node

    return build(0)


def count_nodes(root):
    return 1 + sum(count_nodes(c) for c in root.children)


def collect_nodes(root):
    """Flatten tree to list (pre-order)."""
    result = [root]
    for c in root.children:
        result.extend(collect_nodes(c))
    return result


# Generate our test tree
tree = generate_tree(max_depth=4, branching=3, seed=42)
n = count_nodes(tree)
all_nodes = collect_nodes(tree)

print(f"Knowledge tree: {n} nodes, max depth 4")
print(f"Root: {tree.nid} (info_potential={tree.info_potential:.3f})")
print(f"Tier costs at root: {tuple(f'{c:.1f}' for c in tree.tier_costs)}")
print()

# Show depth distribution
from collections import Counter
depth_dist = Counter(nd.depth for nd in all_nodes)
for d in sorted(depth_dist):
    bar = '#' * depth_dist[d]
    print(f"  depth {d}: {depth_dist[d]:3d} nodes  {bar}")

## The Optimization Problem

From §2.3 and §8.2 of the paper:

> *At each node, the system decides how deeply to engage: Tier 1 (hash check, essentially free),
> Tier 2 (local traversal, moderate), or Tier 3 (LLM call, expensive). The goal is to maximize
> total information gain within a compute budget.*

**Constraint**: You must activate a parent before you can visit its children. This is the tree
structure doing work — it means you can't cherry-pick deep nodes without paying for the path.

**Naive approach**: Tree DP with budget as a state dimension. For each subtree, for each possible
budget allocation, find the optimal tier assignment. Complexity: O(n · B) where B is budget
discretized into steps. For a tree of 100 nodes and budget of 10,000, that's a million states.

**The WQS insight**: Define f(B) = maximum information gain achievable with budget ≤ B.
Claim: f(B) is **concave**. Why?

1. You activate the highest-value nodes first (greedy ordering)
2. Tier upgrades have diminishing marginal gain (30% → 70% → 100% of a node's potential)
3. Deeper tree nodes have lower information potential (by construction — proximity to query matters)

Concavity means we can replace the budget constraint with a per-unit-cost penalty λ and
binary search on λ. The penalized problem decomposes greedily on the tree: O(n) per evaluation,
O(log(1/ε)) binary search iterations. Total: **O(n log(1/ε))**.

## First: Verify Concavity

Before we deploy the trick, let's verify the claim. We'll sweep budget values and solve
each one with a discretized tree DP, then plot f(B). If it's concave, we're in business.

In [ ]:
def tree_dp(root, budget_cap, step=1.0):
    """
    Discretized tree DP: O(n * B/step).
    Returns max gain achievable in subtree within budget.
    """
    B = int(budget_cap / step) + 1

    def solve(node):
        # dp[b] = max gain in this subtree using at most b budget-steps
        dp = [0.0] * B

        # First: combine children via knapsack merge
        child_dps = [solve(c) for c in node.children]
        merged = [0.0] * B
        for cdp in child_dps:
            new_merged = [0.0] * B
            for b1 in range(B):
                if merged[b1] == 0 and b1 > 0:
                    continue  # skip unreachable states (optimization)
                for b2 in range(B - b1):
                    val = merged[b1] + cdp[b2]
                    if val > new_merged[b1 + b2]:
                        new_merged[b1 + b2] = val
            merged = new_merged

        # Then: add this node's tier choice
        for tier in [0, 1, 2, 3]:
            c = int(node.tier_costs[tier] / step)
            g = node.tier_gain(tier)
            if tier > 0:
                # Can use children if active
                for b in range(c, B):
                    val = g + merged[b - c]
                    if val > dp[b]:
                        dp[b] = val
            else:
                # Tier 0: skip node, no children
                for b in range(B):
                    if 0 > dp[b]:  # gain of skipping is 0
                        dp[b] = 0

        # Make dp monotonically non-decreasing (more budget can't hurt)
        for b in range(1, B):
            dp[b] = max(dp[b], dp[b - 1])

        return dp

    dp = solve(root)
    return dp


# Use a smaller tree for the O(n*B) DP (it's expensive)
small_tree = generate_tree(max_depth=3, branching=2, seed=42)
n_small = count_nodes(small_tree)
print(f"Small tree for DP verification: {n_small} nodes")

budget_cap = 500.0
step = 2.0
t0 = time.perf_counter()
dp_result = tree_dp(small_tree, budget_cap, step)
t_dp = time.perf_counter() - t0
print(f"Tree DP completed in {t_dp:.3f}s ({len(dp_result)} budget steps)")
print(f"Max gain at full budget: {dp_result[-1]:.4f}")

In [ ]:
# Verify concavity: check that second differences are non-positive
gains = dp_result
budgets = [i * step for i in range(len(gains))]

# Second differences: f(b+1) - 2f(b) + f(b-1) <= 0 for concavity
second_diffs = []
for i in range(1, len(gains) - 1):
    sd = gains[i + 1] - 2 * gains[i] + gains[i - 1]
    second_diffs.append(sd)

violations = sum(1 for sd in second_diffs if sd > 1e-9)
print(f"Concavity check: {len(second_diffs)} second differences computed")
print(f"Violations (positive second differences): {violations}")
if violations == 0:
    print("f(B) is concave. WQS binary search is valid.")
else:
    # In practice, discretization can cause minor numerical violations
    max_violation = max(second_diffs)
    print(f"Max violation: {max_violation:.2e} (likely discretization noise)")

print()
print("Gain curve (sampled):")
sample_points = [0, 10, 25, 50, 100, 150, 200, 250]
for b in sample_points:
    idx = min(b, len(gains) - 1)
    bar = '#' * int(gains[idx] * 40 / max(max(gains), 0.001))
    print(f"  B={b*step:6.0f}  gain={gains[idx]:.4f}  {bar}")

## Enter the Aliens

The WQS binary search (named after IOI 2016 "Aliens"):

1. **Remove** the budget constraint
2. **Add** a penalty: each unit of cost incurs a charge of λ
3. **Solve** the unconstrained problem: at each node, pick the tier maximizing `gain - λ·cost`
4. **Binary search** on λ until the total cost matches our budget

Step 3 is O(n) on a tree — just a single post-order traversal with a greedy choice at each node.
The parent constraint (must activate parent to visit children) is handled naturally: if a parent's
best penalized value is negative, it stays at Tier 0, and its entire subtree is pruned.

Step 4 is O(log(1/ε)) iterations.

Total: **O(n · log(1/ε))** vs O(n · B) for the naive DP.

In [ ]:
def solve_penalized(root, lam, weights=(0.4, 0.3, 0.3)):
    """
    Solve the Lagrangian relaxation: max Σ(gain_i - λ·cost_i)
    with tree connectivity constraint.

    O(n) — single traversal, greedy per-node choice.
    """
    total_gain = 0.0
    total_cost = 0.0
    active_count = 0
    assignments = {}

    def visit(node, parent_active):
        nonlocal total_gain, total_cost, active_count

        if not parent_active:
            # Parent is off → entire subtree is off
            assignments[node.nid] = 0
            for c in node.children:
                visit(c, False)
            return

        # Greedy: pick tier maximizing gain - λ·cost
        best_tier = 0
        best_value = 0.0  # Tier 0 has value 0

        for t in [1, 2, 3]:
            g = node.tier_gain(t, weights)
            c = node.tier_costs[t]
            val = g - lam * c
            if val > best_value:
                best_value = val
                best_tier = t

        assignments[node.nid] = best_tier
        total_gain += node.tier_gain(best_tier, weights)
        total_cost += node.tier_costs[best_tier]
        if best_tier > 0:
            active_count += 1

        # Children can only be visited if this node is active
        for c in node.children:
            visit(c, best_tier > 0)

    visit(root, True)  # Root's "parent" is always active
    return total_gain, total_cost, active_count, assignments


def wqs_optimize(root, budget, weights=(0.4, 0.3, 0.3), eps=1e-12, max_iter=200):
    """
    WQS binary search: find λ such that the penalized solution uses ≈ budget.

    Returns (gain, cost, active_count, assignments, lambda, iterations).
    """
    lo, hi = 0.0, 1.0  # λ range (gain and cost are both O(1) per node)
    best = None
    iters = 0

    for i in range(max_iter):
        iters = i + 1
        if hi - lo < eps:
            break
        lam = (lo + hi) / 2
        gain, cost, active, assign = solve_penalized(root, lam, weights)

        if cost > budget:
            lo = lam   # increase penalty → reduce spending
        else:
            hi = lam   # decrease penalty → allow more spending
            best = (gain, cost, active, assign, lam)

    if best is None:
        # Budget is too small for any activation
        gain, cost, active, assign = solve_penalized(root, hi, weights)
        best = (gain, cost, active, assign, hi)

    return (*best, iters)


print("WQS solver loaded.")

In [ ]:
# Run WQS on the FULL tree (not the small one — WQS doesn't care about size)
budget = 200.0

t0 = time.perf_counter()
gain, cost, active, assign, lam, iters = wqs_optimize(tree, budget)
t_wqs = time.perf_counter() - t0

print(f"=== WQS Binary Search ===")
print(f"Tree size:       {n} nodes")
print(f"Budget:          {budget:.0f}")
print(f"Optimal λ:       {lam:.6f}")
print(f"Total gain:      {gain:.4f}")
print(f"Total cost:      {cost:.1f} / {budget:.0f}")
print(f"Active nodes:    {active} / {n}")
print(f"Iterations:      {iters}")
print(f"Wall time:       {t_wqs*1000:.2f} ms")
print()

# Show tier distribution
tier_counts = Counter(assign.values())
for t in sorted(tier_counts):
    label = ['skip', 'hash_check', 'local_trav', 'llm_call'][t]
    print(f"  Tier {t} ({label:10s}): {tier_counts[t]} nodes")

In [ ]:
# Verify WQS against the naive DP on the small tree
gain_wqs_s, cost_wqs_s, _, _, _, iters_s = wqs_optimize(small_tree, budget)
dp_gain_at_budget = dp_result[min(int(budget / step), len(dp_result) - 1)]

print(f"=== Verification: WQS vs Naive DP (small tree, {n_small} nodes) ===")
print(f"Naive DP gain:   {dp_gain_at_budget:.4f}  ({t_dp*1000:.1f} ms)")
print(f"WQS gain:        {gain_wqs_s:.4f}  (<1 ms, {iters_s} iterations)")
print(f"Gap:             {abs(dp_gain_at_budget - gain_wqs_s):.6f}")
print()
if abs(dp_gain_at_budget - gain_wqs_s) < 0.05:
    print("Results match (within discretization tolerance).")
    print("WQS finds the same optimum in O(n log V) vs O(n*B).")
else:
    print("Note: gap may be due to DP discretization or tie-breaking.")
    print("The Lagrangian relaxation is a bound; exact recovery depends on concavity.")

## Now the Tunables

The paper identifies several parameters that need calibration (§2.2, §2.5, §9):

| Tunable | Paper section | What it controls | WQS constraint |
|---------|--------------|------------------|----------------|
| Depth decay rate | §8.2 | How fast node scores cool | Warmth budget (how many nodes stay hot) |
| Rewarm rate | §8.2 | How much a query boosts a node | Activation budget |
| Tier thresholds | §2.3 | When to promote Tier 1→2→3 | Promotion count |
| Scoring weights | §2.2 | recency vs centrality vs info_gain | Weight budget (sum = 1) |

Each of these is a budget constraint on a concave objective. Each gets its own λ.

### Tunable 1: Depth Decay ("How many nodes to keep warm")

From §8.2: *"Depth score behaves like temperature. Recently traversed or information-rich
nodes are warm... The scoring function functions analogously to a cooling law."*

The decay rate controls how aggressively the system prunes cold nodes. Too aggressive →
you miss information. Too lax → you waste compute on stale nodes. WQS: binary search
on the warmth penalty to hit a target active-node count.

In [ ]:
def apply_decay(root, decay_rate, current_time=1.0):
    """Apply exponential decay to recency scores. Returns a modified copy."""
    import copy
    decayed = copy.deepcopy(root)

    def decay(node):
        # Recency decays exponentially from last access
        time_since = current_time * (1.0 - node.recency)  # higher recency = more recent
        node.recency = math.exp(-decay_rate * time_since)
        for c in node.children:
            decay(c)

    decay(decayed)
    return decayed


def wqs_warmth(root, target_active, eps=1e-12, max_iter=200):
    """
    WQS on active-node count: find the penalty λ_warm such that
    the optimal solution activates exactly target_active nodes.

    Each activation incurs penalty λ_warm (on top of tier costs).
    """
    lo, hi = 0.0, 5.0
    best = None

    for _ in range(max_iter):
        if hi - lo < eps:
            break
        lam_warm = (lo + hi) / 2

        # Modified penalized solve: extra per-activation cost
        total_gain = 0.0
        total_cost = 0.0
        active = 0
        assignments = {}

        def visit(node, parent_active):
            nonlocal total_gain, total_cost, active
            if not parent_active:
                assignments[node.nid] = 0
                for c in node.children:
                    visit(c, False)
                return

            best_tier = 0
            best_val = 0.0
            for t in [1, 2, 3]:
                g = node.tier_gain(t)
                # Penalty for being active at all
                val = g - lam_warm
                if val > best_val:
                    best_val = val
                    best_tier = t

            assignments[node.nid] = best_tier
            total_gain += node.tier_gain(best_tier)
            total_cost += node.tier_costs[best_tier]
            if best_tier > 0:
                active += 1
            for c in node.children:
                visit(c, best_tier > 0)

        visit(root, True)

        if active > target_active:
            lo = lam_warm
        else:
            hi = lam_warm
            best = (total_gain, total_cost, active, assignments, lam_warm)

    return best


# Show how decay rate interacts with optimal warmth
print("Decay rate → Optimal active nodes at various warmth targets")
print(f"{'decay':>8s}  {'target=10':>12s}  {'target=20':>12s}  {'target=30':>12s}")
print("-" * 52)

for decay_rate in [0.1, 0.5, 1.0, 2.0, 5.0]:
    decayed = apply_decay(tree, decay_rate)
    results = []
    for target in [10, 20, 30]:
        r = wqs_warmth(decayed, target)
        if r:
            results.append(f"{r[2]:3d} @ λ={r[4]:.3f}")
        else:
            results.append("  infeasible")
    print(f"{decay_rate:8.1f}  {results[0]:>12s}  {results[1]:>12s}  {results[2]:>12s}")

### Tunable 2: Scoring Weights

From §2.2: *"The scoring function takes into account: recency, information gain,
structural centrality, query relevance."*

The weights (w_r, w_c, w_g) with w_r + w_c + w_g = 1 determine which nodes look "important."
Different weight configurations lead to different optimal tier assignments. We can
search the weight simplex efficiently by parameterizing it and using WQS at each point.

In [ ]:
# Sweep the weight simplex: for each weight config, run WQS to find optimal allocation
budget = 200.0
results = []

# Sample weight simplex (w_recency, w_centrality, w_info_gain) with w_r + w_c + w_g = 1
weight_configs = []
steps = 10
for wr in range(0, steps + 1):
    for wc in range(0, steps + 1 - wr):
        wg = steps - wr - wc
        weight_configs.append((wr / steps, wc / steps, wg / steps))

t0 = time.perf_counter()
for w in weight_configs:
    gain, cost, active, assign, lam, iters = wqs_optimize(tree, budget, weights=w)
    results.append((w, gain, cost, active, lam))
t_sweep = time.perf_counter() - t0

# Find best weight config
results.sort(key=lambda r: r[1], reverse=True)

print(f"Weight simplex sweep: {len(weight_configs)} configs in {t_sweep*1000:.1f} ms")
print(f"({t_sweep*1000/len(weight_configs):.2f} ms per config — each is an O(n log V) WQS solve)")
print()
print("Top 5 weight configurations:")
print(f"{'w_recency':>10s} {'w_central':>10s} {'w_info':>10s} {'gain':>8s} {'cost':>8s} {'active':>7s}")
print("-" * 60)
for w, gain, cost, active, lam in results[:5]:
    print(f"{w[0]:10.2f} {w[1]:10.2f} {w[2]:10.2f} {gain:8.4f} {cost:8.1f} {active:7d}")

print()
print("Bottom 5 weight configurations:")
print(f"{'w_recency':>10s} {'w_central':>10s} {'w_info':>10s} {'gain':>8s} {'cost':>8s} {'active':>7s}")
print("-" * 60)
for w, gain, cost, active, lam in results[-5:]:
    print(f"{w[0]:10.2f} {w[1]:10.2f} {w[2]:10.2f} {gain:8.4f} {cost:8.1f} {active:7d}")

## The Decomposition Theorem: All of Them at Once

Here's where the tree structure really pays off.

Each tunable adds a constraint. Each constraint adds a Lagrangian penalty λ_k.
The inner solver — the greedy tree traversal — handles all penalties simultaneously:
at each node, pick the tier maximizing

$$\text{gain}(i, t) - \lambda_{\text{cost}} \cdot \text{cost}(i, t) - \lambda_{\text{warm}} \cdot \mathbb{1}[t > 0] - \lambda_{\text{tier3}} \cdot \mathbb{1}[t = 3]$$

This is still O(n) per evaluation. Each binary search adds a log factor.
With k tunables: **O(n · (log V)^k)**.

Compare to grid search at equal precision. Binary search halves the interval each step,
so P iterations gives precision range/2^P. Grid search needs V = range/ε points per
dimension for the same precision ε. The comparison:

| Method | Inner solves (k constraints) | Precision per dim |
|--------|------------------------------|-------------------|
| Grid search | V^k | range/V |
| Nested WQS | P^k | range/2^P |

For equal precision (range/V = range/2^P → V = 2^P):
- Grid: (2^P)^k = 2^(Pk) inner solves
- WQS: P^k inner solves
- Ratio: 2^(Pk) / P^k — **exponential in P**

Concretely, for k=3 constraints and P=15 iterations per level:
- WQS: 15³ = 3,375 inner solves, precision ~0.00015 per dimension
- Grid at same precision: 33,333³ ≈ 3.7 × 10^13 inner solves

That's not a constant-factor speedup. It's a different complexity class.

In [ ]:
def optimize_joint(root, cost_budget, warmth_target, max_tier3,
                   weights=(0.4, 0.3, 0.3), precision_iters=40):
    """
    Joint optimization over three constraints simultaneously:
    1. Total cost ≤ cost_budget        (λ_cost)
    2. Active nodes ≤ warmth_target    (λ_warm)
    3. Tier-3 nodes ≤ max_tier3        (λ_tier3)

    Nested WQS: O(n · iters^3).
    """
    eval_count = [0]

    def solve_inner(root, lam_cost, lam_warm, lam_tier3):
        """O(n) greedy solve with all three penalties."""
        eval_count[0] += 1
        total_gain = 0.0
        total_cost = 0.0
        active = 0
        tier3_count = 0
        assignments = {}

        def visit(node, parent_on):
            nonlocal total_gain, total_cost, active, tier3_count
            if not parent_on:
                assignments[node.nid] = 0
                for c in node.children:
                    visit(c, False)
                return

            best_t, best_v = 0, 0.0
            for t in [1, 2, 3]:
                g = node.tier_gain(t, weights)
                penalty = lam_cost * node.tier_costs[t]
                if t > 0:
                    penalty += lam_warm
                if t == 3:
                    penalty += lam_tier3
                val = g - penalty
                if val > best_v:
                    best_v = val
                    best_t = t

            assignments[node.nid] = best_t
            total_gain += node.tier_gain(best_t, weights)
            total_cost += node.tier_costs[best_t]
            if best_t > 0:
                active += 1
            if best_t == 3:
                tier3_count += 1
            for c in node.children:
                visit(c, best_t > 0)

        visit(root, True)
        return total_gain, total_cost, active, tier3_count, assignments

    P = precision_iters
    best_result = None
    best_gain = -1

    # Outer: binary search on λ_tier3
    lo3, hi3 = 0.0, 5.0
    for _ in range(P):
        lam3 = (lo3 + hi3) / 2

        # Middle: binary search on λ_warm
        lo_w, hi_w = 0.0, 5.0
        inner_best = None
        for _ in range(P):
            lam_w = (lo_w + hi_w) / 2

            # Inner: binary search on λ_cost
            lo_c, hi_c = 0.0, 1.0
            for _ in range(P):
                lam_c = (lo_c + hi_c) / 2
                gain, cost, act, t3, assign = solve_inner(root, lam_c, lam_w, lam3)

                if cost > cost_budget:
                    lo_c = lam_c
                else:
                    hi_c = lam_c
                    if gain > best_gain and act <= warmth_target and t3 <= max_tier3:
                        best_gain = gain
                        best_result = (gain, cost, act, t3, assign, lam_c, lam_w, lam3)
                    inner_best = (act, t3)

            if inner_best and inner_best[0] > warmth_target:
                lo_w = lam_w
            else:
                hi_w = lam_w

        if inner_best and inner_best[1] > max_tier3:
            lo3 = lam3
        else:
            hi3 = lam3

    return best_result, eval_count[0]


# Joint optimization: cost ≤ 200, active ≤ 25, tier-3 ≤ 3
P = 15
t0 = time.perf_counter()
result, evals = optimize_joint(tree, cost_budget=200, warmth_target=25, max_tier3=3,
                                precision_iters=P)
t_joint = time.perf_counter() - t0

if result:
    gain, cost, active, t3, assign, lc, lw, l3 = result
    tier_dist = Counter(assign.values())

    # For equal-precision grid search: P binary search iterations over range [0,5]
    # gives precision 5/2^P ≈ 0.00015. Grid needs 5/0.00015 ≈ 33,333 points per dim.
    grid_equal_precision = int(5 / (5 / 2**P)) ** 3

    print(f"=== Joint Optimization (3 constraints) ===")
    print(f"Tree size:        {n} nodes")
    print(f"Constraints:      cost≤200, active≤25, tier3≤3")
    print(f"")
    print(f"Optimal gain:     {gain:.4f}")
    print(f"Total cost:       {cost:.1f}")
    print(f"Active nodes:     {active}")
    print(f"Tier-3 nodes:     {t3}")
    print(f"")
    print(f"Lagrange multipliers:")
    print(f"  λ_cost  = {lc:.6f}  (cost penalty per unit)")
    print(f"  λ_warm  = {lw:.6f}  (activation penalty per node)")
    print(f"  λ_tier3 = {l3:.6f}  (Tier-3 penalty per node)")
    print(f"")
    print(f"=== Speedup Analysis ===")
    print(f"WQS inner solves:       {evals:,}  (each is O(n={n}) work)")
    print(f"Precision per dim:      {5/2**P:.6f}")
    print(f"Grid at same precision: {grid_equal_precision:,} inner solves")
    print(f"Speedup (equal prec):   {grid_equal_precision / max(evals, 1):,.0f}x")
    print(f"Wall time:              {t_joint*1000:.1f} ms")
    print(f"")
    print(f"Tier distribution:")
    for t in sorted(tier_dist):
        label = ['skip', 'hash_check', 'local_trav', 'llm_call'][t]
        bar = '#' * tier_dist[t]
        print(f"  Tier {t} ({label:10s}): {tier_dist[t]:3d}  {bar}")
else:
    print("No feasible solution found within constraints.")

## The Fourth Constraint: Scoring Function Hyperparameters

The scoring weights (w_recency, w_centrality, w_info_gain) aren't a budget constraint in
the WQS sense — they're a simplex constraint. But here's the trick: the joint optimizer
above is fast enough to serve as an **inner loop** for a sweep over the weight simplex.

For each weight configuration, we run the joint WQS to find the optimal tier assignment.
The best weight config is the one that achieves the highest gain across all joint constraints.

This is the "all of them" answer: weights are searched on the simplex, and for each
weight vector, the budget/warmth/tier3 constraints are handled by nested WQS.
Total: O(S · n · (log V)^3) where S is the simplex sample count.

In [ ]:
# Full hyperparameter optimization: weights × joint constraints
t0 = time.perf_counter()
full_results = []
total_evals = 0

# Sample weight simplex at resolution 5 (21 points)
simplex_steps = 5
P_full = 12  # iterations per binary search level
for wr in range(0, simplex_steps + 1):
    for wc in range(0, simplex_steps + 1 - wr):
        wg = simplex_steps - wr - wc
        w = (wr / simplex_steps, wc / simplex_steps, wg / simplex_steps)
        # Skip degenerate weights
        if min(w) < 0.05:
            continue

        result, evals = optimize_joint(
            tree, cost_budget=200, warmth_target=25, max_tier3=3,
            weights=w, precision_iters=P_full
        )
        total_evals += evals
        if result:
            full_results.append((w, result))

t_full = time.perf_counter() - t0

# Sort by gain
full_results.sort(key=lambda r: r[1][0], reverse=True)

# Correct speedup: compare inner solves at equal precision, not node visits
S = len(full_results)
grid_per_config = int(5 / (5 / 2**P_full)) ** 3  # grid points for same precision
grid_total = grid_per_config * S
wqs_total = total_evals

print(f"=== Full Hyperparameter Optimization ===")
print(f"Weight configs tested:  {S}")
print(f"Total inner solves:     {wqs_total:,}  (each O(n={n}))")
print(f"Total wall time:        {t_full*1000:.0f} ms")
print(f"")
print(f"=== Honest Speedup Accounting ===")
print(f"WQS precision per dim:  {5/2**P_full:.6f}  ({P_full} binary search iters)")
print(f"Grid at same precision: {grid_per_config:,} solves/config × {S} configs = {grid_total:,} total")
print(f"WQS total:              {wqs_total:,}")
print(f"Speedup (equal prec):   {grid_total / max(wqs_total, 1):,.0f}x")
print()

if full_results:
    print("Top 3 configurations:")
    print(f"{'w_r':>6s} {'w_c':>6s} {'w_g':>6s} | {'gain':>7s} {'cost':>7s} {'active':>7s} {'T3':>4s} | {'λ_cost':>8s} {'λ_warm':>8s} {'λ_T3':>8s}")
    print("-" * 80)
    for w, r in full_results[:3]:
        gain, cost, act, t3, _, lc, lw, l3 = r
        print(f"{w[0]:6.2f} {w[1]:6.2f} {w[2]:6.2f} | {gain:7.4f} {cost:7.1f} {act:7d} {t3:4d} | {lc:8.5f} {lw:8.5f} {l3:8.5f}")

    print()
    best_w = full_results[0][0]
    print(f"Optimal weights: recency={best_w[0]:.2f}, centrality={best_w[1]:.2f}, info_gain={best_w[2]:.2f}")

## Fixed-Point Check

From the paper (§2.5): *"A reasoning cycle terminates when running another cycle would
produce the same root hash."*

The optimized configuration should converge faster — the whole point of calibration is
that well-tuned depth scores direct compute toward genuinely uncertain regions.

We simulate this: apply the optimized tier assignments, measure how many re-evaluations
it takes for the assignment to stabilize (no node changes tier). Compare optimized vs
uniform assignment.

In [ ]:
def simulate_convergence(root, initial_assignments, n_cycles=20, noise=0.02, seed=123):
    """
    Simulate reasoning cycles: each cycle adds small noise to recency scores
    (simulating new observations), then re-optimizes. Track how fast assignments
    stabilize (= fixed point).
    """
    import copy
    rng = random.Random(seed)
    current = copy.deepcopy(root)
    prev_assign = dict(initial_assignments)
    history = []

    for cycle in range(n_cycles):
        # Perturb: small noise to recency (new observations arrive)
        all_n = collect_nodes(current)
        for nd in all_n:
            nd.recency = max(0, min(1, nd.recency + rng.gauss(0, noise)))

        # Re-optimize with WQS
        gain, cost, active, assign, lam, iters = wqs_optimize(current, budget=200)

        # Count changed assignments
        changes = sum(1 for nid in assign if assign.get(nid) != prev_assign.get(nid))
        history.append({"cycle": cycle, "changes": changes, "gain": gain,
                       "cost": cost, "active": active})
        prev_assign = dict(assign)

    return history


# Optimized starting point
if full_results:
    best_w, best_r = full_results[0]
    _, _, _, _, opt_assign, _, _, _ = best_r
else:
    _, _, _, opt_assign, _, _ = wqs_optimize(tree, 200)

# Uniform starting point (all nodes at Tier 1)
uniform_assign = {nd.nid: 1 for nd in all_nodes}

hist_opt = simulate_convergence(tree, opt_assign, noise=0.03)
hist_uni = simulate_convergence(tree, uniform_assign, noise=0.03)

print("Convergence comparison: WQS-optimized vs uniform start")
print(f"{'cycle':>6s}  {'opt_changes':>12s} {'opt_gain':>10s}  {'uni_changes':>12s} {'uni_gain':>10s}")
print("-" * 56)
for o, u in zip(hist_opt, hist_uni):
    print(f"{o['cycle']:6d}  {o['changes']:12d} {o['gain']:10.4f}  {u['changes']:12d} {u['gain']:10.4f}")

# Cycles to convergence (changes ≤ 1)
conv_opt = next((h['cycle'] for h in hist_opt if h['changes'] <= 1), len(hist_opt))
conv_uni = next((h['cycle'] for h in hist_uni if h['changes'] <= 1), len(hist_uni))
print(f"\nCycles to near-convergence (≤1 change):")
print(f"  Optimized: {conv_opt}")
print(f"  Uniform:   {conv_uni}")

## What This Means

The paper (§9) lists "calibration of depth scores" as an open problem:

> *"The specific decay functions, rewarming rates, and the interaction between query frequency
> and score stability require empirical work."*

This notebook suggests that the problem is more tractable than it might appear, because
the substrate's own structure provides the optimization landscape:

1. **The Merkle DAG is a tree** (or a forest of trees rooted at query entry points)
2. **Information gain is concave in compute** (diminishing returns on deeper traversal)
3. **Concavity + tree structure = WQS binary search** (aliens trick)
4. **Each tunable maps to a Lagrangian penalty** binary-searched independently
5. **Multiple tunables compose** via nesting, with cost O(n · (log V)^k)

The "if" at the top of this notebook — "if the substrate provides the right abstraction,
optimization should cleanly follow" — appears to hold. The abstraction (content-addressed
tree with tiered costs) IS the structure that WQS binary search needs.

This is a PoC, not a proof. Open questions:
- Real DAGs have shared parents (diamond patterns). How much concavity survives?
- The gain function here is synthetic. Real information gain is noisy and non-stationary.
- WQS gives you the optimal static configuration. The substrate needs adaptive calibration.
- The "speed of truth" bound (§8.3) may interact with optimization lag in ways that matter.

But the structural observation stands: if your data structure is a tree, and your objective
has diminishing returns, you don't need grid search. You need the aliens trick.

---

*Companion to [A Content-Addressed Adaptive Knowledge Substrate for Distributed Epistemic Coordination](joven_knowledge_substrate.md) (Joven, 2026).*
*This notebook is released under CC0 alongside the paper.*